In [1]:
from utils import *
import pandas as pd
import os
import numpy as np

In [2]:
# Load data and map exercise modality

file_path = os.path.join("tables", "exerciseTableForBN_python.parquet")
exerciseTable = pd.read_parquet(file_path)

exerciseTable = map_exercise_modality(exerciseTable, activity_col_name="CleanActivityName")

In [ ]:
plt.plot(exerciseTable['age'], exerciseTable['gender'], 'o')

In [ ]:

colsToRemove = ["PtID", "DeviceDtTm", "UTCDtTm", "ExerciseName", "DistanceValue", "DistanceUnits", "DurationUnits", "EnergyValue",
                 "EnergyUnits", "TmZnOffset", "CleanActivityName", "ACWR", "TimeSinceLastBolus", "LastBolus", 
                 "TimeSinceLastBasal", "LastBasal"]

df = exerciseTable.copy()

df_discrete = discretize_data(df, 
    cv_strategy = "clinical",
    bmi_strategy = "clinical",
    hba1c_strategy = "clinical", 
    glucose_strategy = "clinical", 
    cols_to_remove = colsToRemove)

In [4]:
# Define features and tiers

tiers = {
    'static': ['age', 'gender', 'BMI', 'HbA1c', 'InsSensitivity', 'InsCarbRatio'],
    'pre': ['preExerciseRoc', 'startExerciseGlucoseLevel', 'preExerciseGlucoseCV', 'IOBnorm', 'COBnorm', 'AOB', 'TotalCWL'],
    'exercise': ['MET_min', 'ExerciseModality'],
    'outcome_during': ['exerciseGlucoseRoc'],
    'outcome_post': ['minGlucosePostExercise', 'postExerciseTIR', 'postExerciseGlucoseCV']
}

features_to_drop = ["height", "weight", "DurationValue", "MET", "EnergyPerMinute", "exerciseGlucoseExcursion", "CWLPerDay", "ACWR", "COB", "IOB"]

In [ ]:
# Prepare Data & Check Discretization

df_clean = prepare_data(df_discrete, features_to_drop)

check_discretization(df_clean) # comment if not needed!

In [ ]:
# Analyze Unknown Sources

analyze_unknown_sources(df, df_clean)

In [ ]:
# Visualize Feature Distribution after Discretization

target_feature = "ExerciseModality"

# Plot the Raw Clinical thresholds
plot_single_feature_distribution(df_clean, target_feature, title_prefix="- Clinical Thresholds")


In [ ]:
# Data Driven BN

collapse = False

if(collapse):
    df_clean_collapsed = collapse_sparse_bins(df_clean) 
    learner_data_driven = gum.BNLearner(df_clean_collapsed)
else:
    learner_data_driven = gum.BNLearner(df_clean)

learner_data_driven.useGreedyHillClimbing()
bn_data_driven = learner_data_driven.learnBN()

visualize_network(bn_data_driven, tiers)

In [ ]:
# Visualize Feature Distribution after Discretization and Bins Collapse

target_feature = "preExerciseGlucoseCV"

# Plot the Raw Clinical thresholds
plot_single_feature_distribution(df_clean_collapsed, target_feature, title_prefix="- Clinical Thresholds After Merge")

In [ ]:
# Constrained BN

collapse = True

if(collapse):
    df_clean_collapsed = collapse_sparse_bins(df_clean) # comment if you want to use the original discretization for the constrained model!
    learner_constrained = gum.BNLearner(df_clean_collapsed)
else:
    learner_constrained = gum.BNLearner(df_clean)
    
learner_constrained.useGreedyHillClimbing()
learner_constrained = apply_expert_constraints(learner_constrained, tiers)
bn_constrained = learner_constrained.learnBN()

visualize_network(bn_constrained, tiers)

INFERENCE

In [ ]:
# Launch the interactive inference GUI
gnb.showInference(bn_constrained, size="15")
# 1. Define the evidence (the nodes you want to "click")
# Make sure these strings exactly match your discretized categories
my_evidence = {
    'ExerciseModality': 'Aerobic', 
}

# 2. Pass the evidence directly into the visualizer!
# pyagrum will calculate the math and draw the updated graph
gnb.showInference(bn_constrained, evs=my_evidence, size="15")





In [ ]:

# 1. Create an Inference Engine for your trained network
ie = gum.LazyPropagation(bn_constrained)

# 2. Define a hypothetical patient scenario (The Evidence)
# Make sure the string states perfectly match the names of your discretized bins!
evidence = {
    'MET_min': 'High',               
    'startExerciseGlucoseLevel': 'Hypo Risk', 
    'IOB': 'High'                 
}

ie.setEvidence(evidence)
ie.makeInference()

# 3. Query the Outcome Node
print("--- PREDICTING POST-EXERCISE GLUCOSE ---")
target_node = 'minGlucosePostExercise'
posterior = ie.posterior(target_node)
print(posterior)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pyagrum as gum

# 1. Initialize the Inference Engine with your trained network
ie = gum.LazyPropagation(bn_constrained)

# 2. Define our Evidence Nodes (The variables we want to "fix")
# We extract the exact labels from your network to populate the dropdowns
metMin_states = list(bn_constrained.variableFromName('MET_min').labels())
iob_states = list(bn_constrained.variableFromName('IOB').labels())
start_glucose_states = list(bn_constrained.variableFromName('startExerciseGlucoseLevel').labels())

# We add a 'Not Fixed' option so the network can calculate baseline probabilities
drop_metMin = widgets.Dropdown(options=['Not Fixed'] + metMin_states, description='MET*min (Load):', style={'description_width': 'initial'})
drop_iob = widgets.Dropdown(options=['Not Fixed'] + iob_states, description='IOB (Insulin):', style={'description_width': 'initial'})
drop_glucose = widgets.Dropdown(options=['Not Fixed'] + start_glucose_states, description='Start Glucose:', style={'description_width': 'initial'})

# 3. Define our Target Node (The outcome we want to watch)
target_node = 'minGlucosePostExercise'
target_states = list(bn_constrained.variableFromName(target_node).labels())

# Create an output area for our dynamically updating chart
out_plot = widgets.Output()

# 4. The Core Update Function
# This runs every single time you change a dropdown!
def update_dashboard(*args):
    with out_plot:
        clear_output(wait=True)
        
        # Reset the engine to baseline
        ie.eraseAllEvidence()
        
        # Inject the "Fixed" evidence from the dropdowns
        if drop_metMin.value != 'Not Fixed': 
            ie.addEvidence('MET_min', drop_metMin.value)
        if drop_iob.value != 'Not Fixed': 
            ie.addEvidence('IOB', drop_iob.value)
        if drop_glucose.value != 'Not Fixed': 
            ie.addEvidence('startExerciseGlucoseLevel', drop_glucose.value)
            
        # Calculate the new probabilities!
        ie.makeInference()
        posterior = ie.posterior(target_node)
        
        # Extract the exact math probabilities
        probs = [posterior[i] * 100 for i in range(len(target_states))] # Convert to percentages
        
        # Draw the Bar Chart
        plt.figure(figsize=(8, 4))
        bars = plt.bar(target_states, probs, color='#2ca02c', edgecolor='black')
        plt.ylim(0, 100)
        plt.ylabel('Probability (%)', fontsize=12)
        plt.title(f'Predicted Outcome: {target_node}', fontsize=14, fontweight='bold')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        
        # Add the exact percentage numbers on top of the bars
        for bar in bars:
            yval = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2, yval + 2, f'{yval:.1f}%', ha='center', va='bottom', fontweight='bold')
            
        plt.tight_layout()
        plt.show()

# 5. Connect the dropdowns to the update function
drop_metMin.observe(update_dashboard, names='value')
drop_iob.observe(update_dashboard, names='value')
drop_glucose.observe(update_dashboard, names='value')

# 6. Display the Dashboard!
ui = widgets.VBox([
    widgets.HTML("<h3>T1D Exercise Inference Engine</h3>"),
    widgets.HBox([drop_metMin, drop_iob, drop_glucose]),
    out_plot
])

display(ui)

# Trigger the initial baseline plot
update_dashboard()